# Mark each boneJig to distinguish subject and pose

In [1]:
import pyvista as pv
from trimesh.interfaces import blender
from pathlib import Path
import numpy as np

from phd_helpers.CartilageGeneration import get_trimesh, get_pvmesh

In [ ]:
subs = ['50000R', '50017L', '50034R']
poses = ['pinch_load', 'flexion', 'extension', 'abduction', 'adduction']

mesh_dir = Path('../../../3Dprints/instronJigPipeline/outputs/CA-35T/meshes')

tpm_mesh_paths = list(mesh_dir.glob(f'*/tpm_boneJig*.vtp'))
tpm_mesh_paths = [x for x in tpm_mesh_paths if x.parent.name in subs and x.name.split('g_')[-1].split('-')[0] in poses]
mc1_mesh_paths = list(mesh_dir.glob(f'*/mc1_boneJig*.vtp'))
mc1_mesh_paths = [x for x in mc1_mesh_paths if x.parent.name in subs]

counts = {
    '50000R': 1,
    '50017L': 2,
    '50034R': 3,
    'pinch_load': 1,
    'flexion': 2,
    'extension': 3,
    'abduction': 4,
    'adduction': 5,
}

offsets = {
    'subs': [-8, -9, -10], # Z
    'poses': [(-8, 1), (-9, 1), (-10, 1), (-8, 2), (-9, 2)] # Z, Y
}

indent = 0.3 # embossment indent X
size = 0.7  # embosser size YxZ

# Test

In [160]:
tpm_path = tpm_mesh_paths[14]
tpm_mesh = pv.read(tpm_path)
sub = tpm_path.parent.name
pose = tpm_path.name.split('-')[0].split('g_')[-1]
sub_count = counts[sub]
pose_count = counts[pose]

# Find large ledge upper surface location
mesh_X = tpm_mesh.points[:, 0]
mesh_Z = tpm_mesh.points[:, 2]
upper_X = np.unique(mesh_X[mesh_Z < -10]).max()

# dividing line
line = pv.Box(bounds=(-indent, 5.0-indent, -0.15, 0.15, -1.5, 1.5))
line.translate((upper_X, 0, -9), inplace=True)

print(sub, pose)

50017L extension


In [161]:
pl = pv.Plotter()
pl.add_mesh(tpm_mesh, opacity=1)
#pl.add_points(np.zeros(3), point_size=20, color='black)
for offset_Z in offsets['subs'][:sub_count]:
    dobber = pv.Box(bounds=(-indent, 5.0-indent, -0.5*size, 0.5*size, -0.3*size, 0.3*size))
    dobber.translate((upper_X, -1, offset_Z), inplace=True)
    pl.add_mesh(dobber, color='gray')
for offset_Z, offset_Y in offsets['poses'][:pose_count]:
    dobber = pv.Box(bounds=(-indent, 5.0-indent, -0.5*size, 0.5*size, -0.5*size, 0.5*size))
    dobber.translate((upper_X, offset_Y, offset_Z), inplace=True)
    pl.add_mesh(dobber, color='gray')
pl.add_mesh(line, color='gray')
pl.show()

Widget(value='<iframe src="http://localhost:58246/index.html?ui=P_0x35cc00590_26&reconnect=auto" class="pyvist…

# Clean code

In [199]:
import trimesh
from phd_helpers.paths import get_boundary

### TPMs

In [187]:
for i, tpm_path in enumerate(tpm_mesh_paths):
    tpm_mesh = pv.read(tpm_path)    
    sub = tpm_path.parent.name
    pose = tpm_path.name.split('-')[0].split('g_')[-1]
    sub_count = counts[sub]
    pose_count = counts[pose]
    print(i, sub, pose)

    # Find large ledge upper surface location
    mesh_X = tpm_mesh.points[:, 0]
    mesh_Z = tpm_mesh.points[:, 2]
    upper_X = np.unique(mesh_X[mesh_Z < -10]).max()

    # emboss
    #size = sizes[sub][pose]

    line = pv.Box(bounds=(-indent, indent, -0.15, 0.15, -1.5, 1.5))
    line.translate((upper_X, 0, -9), inplace=True)

    cutters = line
    for offset_Z in offsets['subs'][:sub_count]:
        dobber = pv.Box(bounds=(-indent, indent, -0.5*size, 0.5*size, -0.3*size, 0.3*size))
        dobber.translate((upper_X, -1, offset_Z), inplace=True)
        cutters += dobber

    for offset_Z, offset_Y in offsets['poses'][:pose_count]:
        dobber = pv.Box(bounds=(-indent, indent, -0.5*size, 0.5*size, -0.5*size, 0.5*size))
        dobber.translate((upper_X, offset_Y, offset_Z), inplace=True)
        cutters += dobber

    # bool
    tpm_mesh = get_pvmesh(blender.boolean([get_trimesh(tpm_mesh), get_trimesh(cutters.triangulate())], operation='difference'))

    # fix
    tpm_mesh.fill_holes(0.01, inplace=True)
    if sub == '50017L' and pose == 'abduction':
        tpm_mesh = pv.PolyData(tpm_mesh.points, np.r_[tpm_mesh.faces, [3, 139, 150, 160]]) # only for 50017L_abduction
    tpm_tmesh = get_trimesh(tpm_mesh)
    trimesh.repair.fix_normals(tpm_tmesh)
    tpm_mesh = get_pvmesh(tpm_tmesh)
    print(tpm_tmesh.is_watertight, tpm_tmesh.is_volume, tpm_tmesh.is_winding_consistent)
    print(tpm_mesh.is_manifold)
    #tpm_mesh.save(f'boneJigs_labelled/{sub}_tpm_{pose}.vtp')

0 50017L abduction
True True True
True


### MC1s

In [207]:
for i, mc1_path in enumerate(mc1_mesh_paths):
    mc1_mesh = pv.read(mc1_path)    
    sub = mc1_path.parent.name
    sub_count = counts[sub]
    print(i, sub)

    # Find large ledge upper surface location
    mesh_X = mc1_mesh.points[:, 0]
    mesh_Z = mc1_mesh.points[:, 2]
    upper_X = np.unique(mesh_X[mesh_Z < -10]).max() -2.5

    # emboss
    #size = sizes[sub][pose]

    line = pv.Box(bounds=(-indent, indent, -0.15, 0.15, -1.5, 1.5))
    line.translate((upper_X, 0, -9), inplace=True)

    cutters = line
    for offset_Z in offsets['subs'][:sub_count]:
        dobber = pv.Box(bounds=(-indent, indent, -0.5*size, 0.5*size, -0.3*size, 0.3*size))
        dobber.translate((upper_X, -1, offset_Z), inplace=True)
        cutters += dobber

    # bool
    mc1_mesh = get_pvmesh(blender.boolean([get_trimesh(mc1_mesh), get_trimesh(cutters.triangulate())], operation='difference'))

    # fix
    mc1_mesh.fill_holes(0.01, inplace=True)
    if sub == '50034R':
        mc1_mesh = pv.PolyData(mc1_mesh.points, np.r_[mc1_mesh.faces, [3, 1015, 1012, 1018]]) # only for 50034R
    mc1_tmesh = get_trimesh(mc1_mesh)
    trimesh.repair.fix_normals(mc1_tmesh)
    mc1_mesh = get_pvmesh(mc1_tmesh)
    print(mc1_tmesh.is_watertight, mc1_tmesh.is_volume, mc1_tmesh.is_winding_consistent)
    print(mc1_mesh.is_manifold)
    #mc1_mesh.save(f'boneJigs_labelled/{sub}_mc1.vtp')

0 50000R
True True True
True
1 50034R
True True True
True
2 50017L
True True True
True


In [203]:
mc1_mesh['point_id'] = np.arange(mc1_mesh.n_points)
edges = get_boundary(mc1_mesh)
edges['point_id']

pyvista_ndarray([1015, 1012, 1018])

In [208]:
pl = pv.Plotter()
pl.add_mesh(mc1_mesh)
pl.show()

Context leak detected, CoreAnalytics returned false


Widget(value='<iframe src="http://localhost:58246/index.html?ui=P_0x35b1c7ce0_32&reconnect=auto" class="pyvist…

# Save 3mf's

In [226]:
import pyvista as pv
import zipfile

def mesh_files_to_multi_part_3mf(mesh_files, output_3mf_path):
    """Convert multiple STL files to a single 3MF with separate objects
    
    Args:
        mesh_files: List of tuples [(mesh_path, object_name), ...]
        output_3mf_path: Output 3MF file path
    """
    model_xml = '''<?xml version="1.0" encoding="UTF-8"?>
<model unit="millimeter" xmlns="http://schemas.microsoft.com/3dmanufacturing/core/2015/02">
  <resources>
'''
    
    build_items = []
    
    for obj_id, (mesh_path, mesh_name) in enumerate(mesh_files, 1):
        stl_mesh = pv.read(mesh_path)
        
        model_xml += f'    <object id="{obj_id}" type="model" name="{mesh_name}">\n'
        model_xml += '      <mesh>\n        <vertices>\n'
        
        vertex_count = 0
        triangles = []
        
        # Add vertices and build triangles
        for triangle in stl_mesh.points[stl_mesh.faces.reshape(-1, 4)[:, 1:]]:
            triangle_indices = []
            for vertex in triangle:
                model_xml += f'          <vertex x="{vertex[0]:.6f}" y="{vertex[1]:.6f}" z="{vertex[2]:.6f}"/>\n'
                triangle_indices.append(vertex_count)
                vertex_count += 1
            triangles.append(triangle_indices)
        
        model_xml += '        </vertices>\n        <triangles>\n'
        
        for triangle in triangles:
            model_xml += f'          <triangle v1="{triangle[0]}" v2="{triangle[1]}" v3="{triangle[2]}"/>\n'
        
        model_xml += '        </triangles>\n      </mesh>\n    </object>\n'
        build_items.append(str(obj_id))
    
    # Add build section
    model_xml += '  </resources>\n  <build>\n'
    for obj_id in build_items:
        model_xml += f'    <item objectid="{obj_id}"/>\n'
    model_xml += '  </build>\n</model>'
    
    # Create the 3MF file (ZIP archive)
    with zipfile.ZipFile(output_3mf_path, 'w', zipfile.ZIP_DEFLATED) as zip_file:
        # Add the main model file
        zip_file.writestr('3D/3dmodel.model', model_xml)
        
        # Add required content types
        content_types = '''<?xml version="1.0" encoding="UTF-8"?>
<Types xmlns="http://schemas.openxmlformats.org/package/2006/content-types">
  <Default Extension="rels" ContentType="application/vnd.openxmlformats-package.relationships+xml"/>
  <Default Extension="model" ContentType="application/vnd.ms-package.3dmanufacturing-3dmodel+xml"/>
</Types>'''
        zip_file.writestr('[Content_Types].xml', content_types)
        
        # Add relationships
        relationships = '''<?xml version="1.0" encoding="UTF-8"?>
<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">
  <Relationship Target="/3D/3dmodel.model" Id="rel-1" Type="http://schemas.microsoft.com/3dmanufacturing/2013/01/3dmodel"/>
</Relationships>'''
        zip_file.writestr('_rels/.rels', relationships)


In [228]:
mesh_dir = Path('../../../3Dprints/instronJigPipeline/outputs/CA-35T/meshes')

subs = ['50000R', '50017L', '50034R']
poses = ['pinch_load', 'flexion', 'extension', 'abduction', 'adduction']

# tpm
for sub in subs:
    for pose in poses:
        cart_path = list(mesh_dir.glob(f'{sub}/tpm_cartilage_{pose}*.vtp'))[0]
        bone_path = f'boneJigs_labelled/{sub}_tpm_{pose}.vtp'

        # save to .3mf file
        mesh_list = [
            (bone_path, 'boneJig'),
            (cart_path, 'cartilage')
        ]
        mesh_files_to_multi_part_3mf(mesh_list, f'boneJigs_labelled/{sub}_tpm_{pose}.3mf')

# mc1
for sub in subs:
    for pose in poses:
        cart_path = list(mesh_dir.glob(f'{sub}/mc1_cartilage-35T.vtp'))[0]
        bone_path = f'boneJigs_labelled/{sub}_mc1.vtp'

        # save to .3mf file
        mesh_list = [
            (bone_path, 'boneJig'),
            (cart_path, 'cartilage')
        ]
        mesh_files_to_multi_part_3mf(mesh_list, f'boneJigs_labelled/{sub}_mc1.3mf')

### Verify 3mf's

In [244]:
import trimesh
mesh_paths = sorted(list(Path('boneJigs_labelled').glob('*.3mf')))
tmesh = trimesh.load(mesh_paths[13])

print("Loaded geometry keys:", list(tmesh.geometry.keys()))
tmesh.show()

Loaded geometry keys: ['boneJig', 'cartilage']
